In [ ]:
"""
Linear Regression Analysis with Multiple Test-Train Splits
Calculates metrics for 90% test sizes
Compares against Baseline (predicting mean)
"""

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("ENHANCED LINEAR REGRESSION WITH 90/10 TEST SPLITS")
print("="*70)


ENHANCED LINEAR REGRESSION WITH 90/10 TEST SPLITS


In [ ]:
# Load datasets
full_data = pd.read_csv('fullv2.csv')
external_data = pd.read_csv('externalv2.csv')


In [ ]:
# Merge data
dataset = full_data.merge(external_data, left_on='match_id', right_on='match_id', how='inner')



In [ ]:
# Create response variable
data = dataset.copy()
data['goal_diff'] = data['homeScore'] - data['awayScore']

In [ ]:
# Select EXACTLY 8 explanatory variables (all pre-match available)
feature_names = [
    'home_Possession',        # 1. Home team ball possession (%)
    'away_Possession',        # 2. Away team ball possession (%)
    'home_xG',                # 3. Home expected goals
    'away_xG',                # 4. Away expected goals
    'betting_odds_home',      # 5. Pre-match betting odds for home
    'referee_strictness',     # 6. Referee strictness rating
    'travel_distance_km',     # 7. Away team travel distance (km)
    'social_sentiment_home'   # 8. Home team social sentiment
]

final_df = data[feature_names + ['goal_diff']].dropna()
X = final_df[feature_names].copy()
y = final_df['goal_diff'].copy()

print(f"\nTotal dataset: {len(X)} matches\n")



Total dataset: 104 matches



In [ ]:
# ============================================================================
# TEST MULTIPLE SPLITS: 70%, 80%, 90%
# ============================================================================

results_summary = []

for test_size in [0.30, 0.20, 0.10]:  # 30%, 20%, 10% test = 70%, 80%, 90% train
    test_pct = int((1 - test_size) * 100)

print(f"\n{'='*70}")
print(f"TEST SIZE: {test_pct}% TRAINING - {int(test_size*100)}% TEST")
print(f"{'='*70}")

X_scaled = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=test_size, random_state=42)

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")



TEST SIZE: 90% TRAINING - 10% TEST
Train samples: 93, Test samples: 11


In [ ]:
# ========================================================================
# MLP PERFORMANCE (Your Linear Regression Model)
# ========================================================================
print(f"\nMLP PERFORMANCE:")
print("-" * 70)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred_test = model.predict(X_test)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred_test)
mse = mean_squared_error(y_test, y_pred_test)
rmse = np.sqrt(mse)
rmse_normalized = rmse / y_test.std()  # Normalized by test set std
r2 = r2_score(y_test, y_pred_test)

intercept = model.intercept_
coef = model.coef_  # Array of coefficients
print(f"Intercept: {intercept:.6f}")
print(f"Coefficients: {[f'{c:.6f}' for c in coef]}")
print(f"\nMAE: {mae:.6f}")
print(f"MSE: {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"RMSE (Normalised): {rmse_normalized:.6f}")
print(f"R²: {r2:.6f}")



MLP PERFORMANCE:
----------------------------------------------------------------------
Intercept: 0.452640
Coefficients: ['0.037156', '-0.102599', '0.794839', '-1.065534', '-0.245516', '-0.164406', '-0.114504', '-0.158255']

MAE: 1.433237
MSE: 2.685348
RMSE: 1.638703
RMSE (Normalised): 0.812003
R²: 0.274716


In [ ]:
# ========================================================================
# BASELINE PERFORMANCE (Predict Mean)
# ========================================================================
print(f"\nBASELINE PERFORMANCE (Predicting Mean):")
print("-" * 70)

# Baseline: always predict the mean of training set
y_pred_baseline = np.full_like(y_test, y_train.mean())

mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
mse_baseline = mean_squared_error(y_test, y_pred_baseline)
rmse_baseline = np.sqrt(mse_baseline)
rmse_baseline_normalized = rmse_baseline / y_test.std()
r2_baseline = r2_score(y_test, y_pred_baseline)

print(f"MAE: {mae_baseline:.6f}")
print(f"MSE: {mse_baseline:.6f}")
print(f"RMSE: {rmse_baseline:.6f}")
print(f"RMSE (Normalised): {rmse_baseline_normalized:.6f}")
print(f"R²: {r2_baseline:.6f}")



BASELINE PERFORMANCE (Predicting Mean):
----------------------------------------------------------------------
MAE: 1.545455
MSE: 3.909091
RMSE: 1.977142
RMSE (Normalised): 0.979705
R²: -0.055804


In [ ]:
# ========================================================================
# IMPROVEMENT OVER BASELINE
# ========================================================================
print(f"\nIMPROVEMENT OVER BASELINE:")
print("-" * 70)

mae_improvement = ((mae_baseline - mae) / mae_baseline) * 100
rmse_improvement = ((rmse_baseline - rmse) / rmse_baseline) * 100
r2_improvement = r2 - r2_baseline

print(f"MAE Improvement: {mae_improvement:.2f}%")
print(f"RMSE Improvement: {rmse_improvement:.2f}%")
print(f"R² Improvement: {r2_improvement:.6f}")

# Store results
results_summary.append({
     'Test Size': f"{test_pct}% Train / {int(test_size*100)}% Test",
     'Intercept': intercept,
     'MAE (MLP)': mae,
     'MSE (MLP)': mse,
     'RMSE (MLP)': rmse,
     'RMSE_Norm (MLP)': rmse_normalized,
     'R² (MLP)': r2,
     'MAE (Baseline)': mae_baseline,
     'MSE (Baseline)': mse_baseline,
     'RMSE (Baseline)': rmse_baseline,
     'RMSE_Norm (Baseline)': rmse_baseline_normalized,
     'R² (Baseline)': r2_baseline,
    })



IMPROVEMENT OVER BASELINE:
----------------------------------------------------------------------
MAE Improvement: 7.26%
RMSE Improvement: 17.12%
R² Improvement: 0.330520


In [ ]:
# ============================================================================
# SUMMARY TABLE
# ============================================================================
print("\n" + "="*70)
print("SUMMARY TABLE FOR YOUR REPORT")
print("="*70)

summary_df = pd.DataFrame(results_summary)
print("\n" + summary_df.to_string(index=False))

# Save to CSV for easy viewing
summary_df.to_csv('model_comparison_multiple_splits.csv', index=False)
print("\n✓ Saved: model_comparison_multiple_splits.csv")



SUMMARY TABLE FOR YOUR REPORT

           Test Size  Intercept  MAE (MLP)  MSE (MLP)  RMSE (MLP)  RMSE_Norm (MLP)  R² (MLP)  MAE (Baseline)  MSE (Baseline)  RMSE (Baseline)  RMSE_Norm (Baseline)  R² (Baseline)
90% Train / 10% Test    0.45264   1.433237   2.685348    1.638703         0.812003  0.274716        1.545455        3.909091         1.977142              0.979705      -0.055804

✓ Saved: model_comparison_multiple_splits.csv


In [ ]:
# ============================================================================
# FORMATTED OUTPUT FOR YOUR TABLE
# ============================================================================
print("\n" + "="*70)
print("FORMATTED OUTPUT FOR YOUR ASSIGNMENT TABLE")
print("="*70)

for idx, row in summary_df.iterrows():
    print(f"\n{row['Test Size'].upper()}")
    print(f"Intercept: {row['Intercept']:.6f}")
    print(f"\nMLP Performance:")
    print(f"  MAE: {row['MAE (MLP)']:.6f}")
    print(f"  MSE: {row['MSE (MLP)']:.6f}")
    print(f"  RMSE: {row['RMSE (MLP)']:.6f}")
    print(f"  RMSE (Normalised): {row['RMSE_Norm (MLP)']:.6f}")
    print(f"  R²: {row['R² (MLP)']:.6f}")

    print(f"\nBaseline Performance:")
    print(f"  MAE: {row['MAE (Baseline)']:.6f}")
    print(f"  MSE: {row['MSE (Baseline)']:.6f}")
    print(f"  RMSE: {row['RMSE (Baseline)']:.6f}")
    print(f"  RMSE (Normalised): {row['RMSE_Norm (Baseline)']:.6f}")
    print(f"  R²: {row['R² (Baseline)']:.6f}")

print("\n" + "="*70)
print("ANALYSIS COMPLETE ✓")
print("="*70)



FORMATTED OUTPUT FOR YOUR ASSIGNMENT TABLE

90% TRAIN / 10% TEST
Intercept: 0.452640

MLP Performance:
  MAE: 1.433237
  MSE: 2.685348
  RMSE: 1.638703
  RMSE (Normalised): 0.812003
  R²: 0.274716

Baseline Performance:
  MAE: 1.545455
  MSE: 3.909091
  RMSE: 1.977142
  RMSE (Normalised): 0.979705
  R²: -0.055804

ANALYSIS COMPLETE ✓


In [ ]:
# ====================================================================
# MLP (Linear Regression) Complete for remaining Test-Train Splits
# ====================================================================

# Create features
X_features = [
    'home_Possession', 'away_Possession', 'home_xG', 'away_xG',
    'betting_odds_home', 'referee_strictness', 'travel_distance_km', 'social_sentiment_home'
]


feature_names = [
    'home_Possession',        # 1. Home team ball possession (%)
    'away_Possession',        # 2. Away team ball possession (%)
    'home_xG',                # 3. Home expected goals
    'away_xG',                # 4. Away expected goals
    'betting_odds_home',      # 5. Pre-match betting odds for home
    'referee_strictness',     # 6. Referee strictness rating
    'travel_distance_km',     # 7. Away team travel distance (km)
    'social_sentiment_home'   # 8. Home team social sentiment
]

final_df = dataset[X_features + ['goal_diff']].dropna()
X = final_df[X_features].copy()
y = final_df['goal_diff'].copy()

print(f"Total dataset: {len(X)} matches\n")

# Store all results
all_results = []

# Test the three splits: 70%, 80%, 90% training
test_splits = [
    (0.30, "70% Training / 30% Test"),
    (0.20, "80% Training / 20% Test"),
    (0.10, "90% Training / 10% Test")
]

for test_size, split_name in test_splits:
    print("\n" + "="*70)
    print(f"TEST SIZE: {split_name}")
    print("="*70)

    X_scaled = StandardScaler().fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=test_size, random_state=42
    )

    print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

    

Total dataset: 104 matches


TEST SIZE: 70% Training / 30% Test
Train samples: 72, Test samples: 32

TEST SIZE: 80% Training / 20% Test
Train samples: 83, Test samples: 21

TEST SIZE: 90% Training / 10% Test
Train samples: 93, Test samples: 11


In [ ]:
# ====================================================================
# MLP (Linear Regression) Performance
# ====================================================================
model = LinearRegression()
model.fit(X_train, y_train)
y_pred_test = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_test)
mse = mean_squared_error(y_test, y_pred_test)
rmse = np.sqrt(mse)
rmse_normalized = rmse / y_test.std()
r2 = r2_score(y_test, y_pred_test)
intercept = model.intercept_

print(f"\nMLP PERFORMANCE:")
print(f"  Intercept: {intercept:.6f}")
print(f"  MAE: {mae:.6f}")
print(f"  MSE: {mse:.6f}")
print(f"  RMSE: {rmse:.6f}")
print(f"  RMSE (Normalised): {rmse_normalized:.6f}")
print(f"  R²: {r2:.6f}")

   


MLP PERFORMANCE:
  Intercept: 0.452640
  MAE: 1.433237
  MSE: 2.685348
  RMSE: 1.638703
  RMSE (Normalised): 0.812003
  R²: 0.274716


In [ ]:
# ====================================================================
# Baseline Performance (Predicting Mean)
# ====================================================================
y_pred_baseline = np.full_like(y_test, y_train.mean(), dtype=float)

mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
mse_baseline = mean_squared_error(y_test, y_pred_baseline)
rmse_baseline = np.sqrt(mse_baseline)
rmse_baseline_normalized = rmse_baseline / y_test.std()
r2_baseline = r2_score(y_test, y_pred_baseline)

print(f"\nBASELINE PERFORMANCE (Predicting Mean):")
print(f"  MAE: {mae_baseline:.6f}")
print(f"  MSE: {mse_baseline:.6f}")
print(f"  RMSE: {rmse_baseline:.6f}")
print(f"  RMSE (Normalised): {rmse_baseline_normalized:.6f}")
print(f"  R²: {r2_baseline:.6f}")

# Store results
all_results.append({
    'Test Size': split_name,
    'Intercept': intercept,
    'MAE (MLP)': mae,
    'MSE (MLP)': mse,
    'RMSE (MLP)': rmse,
    'RMSE_Norm (MLP)': rmse_normalized,
    'R² (MLP)': r2,
    'MAE (Baseline)': mae_baseline,
    'MSE (Baseline)': mse_baseline,
    'RMSE (Baseline)': rmse_baseline,
    'RMSE_Norm (Baseline)': rmse_baseline_normalized,
    'R² (Baseline)': r2_baseline
})



BASELINE PERFORMANCE (Predicting Mean):
  MAE: 1.501466
  MSE: 3.703339
  RMSE: 1.924406
  RMSE (Normalised): 0.953573
  R²: -0.000232


In [ ]:
# ========================================================================
# Save all results to CSV
# ========================================================================
print("\n" + "="*70)
print("SAVING RESULTS")
print("="*70)

results_df = pd.DataFrame(all_results)
results_df.to_csv('model_comparison_all_splits.csv', index=False)
print("✓ Saved: model_comparison_all_splits.csv")

## Print summary table
print("\n" + "="*70)
print("SUMMARY TABLE")
print("="*70)
print("\n" + results_df.to_string(index=False))



SAVING RESULTS
✓ Saved: model_comparison_all_splits.csv

SUMMARY TABLE

              Test Size  Intercept  MAE (MLP)  MSE (MLP)  RMSE (MLP)  RMSE_Norm (MLP)  R² (MLP)  MAE (Baseline)  MSE (Baseline)  RMSE (Baseline)  RMSE_Norm (Baseline)  R² (Baseline)
90% Training / 10% Test    0.45264   1.433237   2.685348    1.638703         0.812003  0.274716        1.501466        3.703339         1.924406              0.953573      -0.000232
